In [98]:
%%writefile lanczos_core.py
#!/usr/bin/env python3
"""
lanczos_core.py — core library for Section V noisy-data pipeline.
Functions:
  build_mock(Nt=32, Ns=16)
    in  : Nt (int), Ns (int)
    out : C list[mp.matrix 2x2], E list[mpf], Z list[(mpf, mpf)]
    eq  : (71)-(72)
  eig_values_vectors(A)
    in  : A (mp.matrix)
    out : vals [mpf], ER (mp.matrix)
    note: wrapper around mp.eig, tolerant to return shapes
  factor_delta(Delta, method="symmetric")
    in  : Delta (mp.matrix), method in {"symmetric","cholesky","lu"}
    out : (beta, gamma) with Delta = beta * gamma
    eq  : (8), (10)-(11)
  block_lanczos(C, m, method="symmetric")
    in  : C (list[mp.matrix]), m (int), method
    out : dict{alpha, beta, gamma, A, G, B}
    eq  : (15), (17), (20), (21), (22)-(23), (24) corrected, (25), (26)-(27), (28) full, (29)
  assemble_T(alphas, betas, gammas, m, r)
    in  : block lists, m, r
    out : T^(m)  (mp.matrix m*r x m*r)
    eq  : (31)-(32)
"""
from mpmath import mp
mp.dps = 40
def build_mock(Nt=32, Ns=16):
    """Eqs 71-72 demo 2x2 mock correlator from 16 hidden states"""
    E, Z = [], []
    for k in range(Ns):
        Ek = mp.mpf("0.1") * (k + 1)
        if k % 2 == 0:
            z0 = mp.mpf("1.0")
            z1 = mp.mpf("-0.1") if k == 0 else mp.mpf("0.1")
        else:
            z0 = mp.mpf("0.1")
            z1 = mp.mpf("1.0")
        Z.append((z0 / mp.sqrt(2 * Ek), z1 / mp.sqrt(2 * Ek)))
        E.append(Ek)
    C = []
    for t in range(Nt):
        Ct = mp.matrix(2, 2)
        for k in range(Ns):
            ek = mp.exp(-E[k] * t)
            for a in range(2):
                for b in range(2):
                    Ct[a, b] += Z[k][a] * Z[k][b] * ek
        C.append(Ct)
    return C, E, Z
def eig_values_vectors(A):
    E, ER = mp.eig(A)
    try:
        vals = [E[k, 0] for k in range(E.rows)]
    except (TypeError, AttributeError):
        vals = list(E)
    if not hasattr(ER, "rows"):
        ER = mp.matrix(ER)
    return vals, ER
def factor_delta(Delta, method="symmetric"):
    """
    Eq 8: Delta = beta * gamma.
    method="symmetric" Eq. 11 (paper's choice for numerical results)
    method="cholesky" valid only for real positive-definite Delta
    method="lu" valid only for complex non-Hermitian Delta (breaks beta=gamma^T, T^(m) non-Hermitian)
    """
    r = Delta.rows
    if r == 1:
        s = mp.sqrt(abs(mp.re(Delta[0, 0])))
        return mp.matrix([[s]]), mp.matrix([[s]])
    Ds = mp.matrix(r, r)
    for i in range(r):
        for j in range(r):
            Ds[i, j] = mp.re(Delta[i, j] + Delta[j, i]) / 2
    if method == "cholesky":
        try:
            L = mp.cholesky(Ds)
            return L, L.T
        except Exception:
            method = "symmetric"
    if method == "lu":
        try:
            P, L, U = mp.lu(Ds)
            Pinv = mp.inverse(P)
            return Pinv * L, U
        except Exception:
            method = "symmetric"
    vals, V = eig_values_vectors(Ds)
    S = mp.diag([mp.sqrt(abs(mp.re(v))) if abs(mp.re(v)) > mp.mpf("1e-40") else mp.mpf("1e-20") for v in vals])
    Vinv = mp.inverse(V)
    return V * S, S * Vinv
def block_lanczos(C, m, method="symmetric"):
    """Eqs 15-29 with Eq 24 corrected (A1(t+2)) and full Eq 28 B/G terms"""
    Nt, r = len(C), C[0].rows
    if Nt < 2 * m:
        raise ValueError(f"need Nt >= 2m; got Nt={Nt}, m={m}")
    a = [None] * (m + 1); b = [None] * (m + 1); g = [None] * (m + 1)
    A = [None] * (m + 1); G = [None] * (m + 1); B = [None] * (m + 1)
    b1, g1 = factor_delta(C[0], method=method)
    b[1], g[1] = b1, g1
    ib1, ig1 = mp.inverse(b1), mp.inverse(g1)
    A1 = [ib1 * C[t] * ig1 for t in range(Nt)]
    A[1] = A1
    a[1] = A1[1]
    if m == 1:
        return dict(alpha=a, beta=b, gamma=g, A=A, G=G, B=B)
    a1 = a[1]
    b2, g2 = factor_delta(A1[2] - a1 * a1, method=method)
    b[2], g[2] = b2, g2
    ib2, ig2 = mp.inverse(b2), mp.inverse(g2)
    G[2] = [ib2 * (A1[t + 1] - a1 * A1[t]) for t in range(Nt - 1)]
    B[2] = [(A1[t + 1] - A1[t] * a1) * ig2 for t in range(Nt - 1)]
    A2 = [ib2 * (A1[t + 2] + a1 * A1[t] * a1 - (a1 * A1[t + 1] + A1[t + 1] * a1)) * ig2 for t in range(Nt - 2)]
    A[2] = A2
    a[2] = A2[1]
    for j in range(2, m):
        Aj, Gj, Bj, Ajm1 = A[j], G[j], B[j], A[j - 1]
        aj, bj, gj = a[j], b[j], g[j]
        bn, gn = factor_delta(Aj[2] - aj * aj - gj * bj, method=method)
        b[j + 1], g[j + 1] = bn, gn
        ibn, ign = mp.inverse(bn), mp.inverse(gn)
        G[j + 1] = [ibn * (Aj[t + 1] - aj * Aj[t] - gj * Bj[t]) for t in range(len(Aj) - 1)]
        B[j + 1] = [(Aj[t + 1] - Aj[t] * aj - Gj[t] * bj) * ign for t in range(len(Aj) - 1)]
        LA = min(len(Aj) - 2, len(Ajm1), len(Bj) - 1, len(Gj) - 1)
        A[j + 1] = []
        for t in range(LA):
            term = (Aj[t + 2] - (aj * Aj[t + 1] + Aj[t + 1] * aj) + aj * Aj[t] * aj + gj * Ajm1[t] * bj - (gj * Bj[t + 1] + Gj[t + 1] * bj)+ gj * Bj[t] * aj + aj * Gj[t] * bj)
            A[j + 1].append(ibn * term * ign)
        a[j + 1] = A[j + 1][1]
    return dict(alpha=a, beta=b, gamma=g, A=A, G=G, B=B)
def assemble_T(a, b, g, m, r):
    """Eq 32: block-tridiagonal T^(m) from alpha, beta, gamma"""
    T = mp.matrix(m * r, m * r)
    for i in range(m):
        for u in range(r):
            for v in range(r):
                T[i * r + u, i * r + v] = a[i + 1][u, v]
    for i in range(m - 1):
        for u in range(r):
            for v in range(r):
                T[i * r + u, (i + 1) * r + v] = b[i + 2][u, v]
                T[(i + 1) * r + u, i * r + v] = g[i + 2][u, v]
    return T

if __name__ == "__main__":
    print("lanczos_core.py — library module")
    print("Import from krylov_verification.py, hermitian_filter.py, "
          "zcw_filter.py, or noisy_run_all.py.")
    print(f"mp.dps = {mp.dps}")

Overwriting lanczos_core.py


In [99]:
%%writefile krylov_verification.py
#!/usr/bin/env python3
"""
krylov_verification.py
Functions:
  compute_krylov_K(m, r, alphas, betas, gammas)
    eq : (44), (45), (46)
  compute_overlaps_eq56_vs_54(C, m, r, omega, omega_inv, beta_1, gamma_1, KR, KL, N_k)
    in : C, eigenvectors, beta1, gamma1, Krylov coefficients, N_k
    out: ZR56, ZL56, ZR54, ZL54, diff_R, diff_L
    eq : (48), (54), (56)
  verify_block_lanczos()
    eq : (30), (37)  — on noiseless mock, m=8
  test_factorization_options()
    eq : (8), (10)-(11)  — symmetric vs cholesky vs LU
  verify_eq56_vs_54()
    eq : (44)-(46), (48), (54), (56)
"""
from mpmath import mp
from lanczos_core import (build_mock, block_lanczos, assemble_T, eig_values_vectors,)
def compute_krylov_K(m, r, alphas, betas, gammas):
    """Eqs 44-46: Krylov coefficients KR_{t,j}, KL_{t,j}"""
    KR, KL = {}, {}
    for j in range(1, m + 1):
        for t in range(j):
            KR[(t, j)] = mp.zeros(r, r)
            KL[(t, j)] = mp.zeros(r, r)
    for a in range(r):
        for b in range(r):
            if a == b:
                KR[(0, 1)][a, b] = 1
                KL[(0, 1)][a, b] = 1
    if m >= 2:
        g2_inv = mp.inverse(gammas[2])
        b2_inv = mp.inverse(betas[2])
        for a in range(r):
            for b in range(r):
                KR[(1, 2)][a, b] = g2_inv[a, b]
                KL[(1, 2)][a, b] = b2_inv[a, b]
                KR[(0, 2)][a, b] = -sum(alphas[1][a, c] * g2_inv[c, b] for c in range(r))
                KL[(0, 2)][a, b] = -sum(b2_inv[a, c] * alphas[1][c, b] for c in range(r))
    for j in range(2, m):
        gj1_inv = mp.inverse(gammas[j + 1])
        bj1_inv = mp.inverse(betas[j + 1])
        for t in range(j + 1):
            for a in range(r):
                for b in range(r):
                    term = mp.zeros(r, r)
                    if t - 1 >= 0 and (t - 1, j) in KR:
                        term += KR[(t - 1, j)]
                    if (t, j) in KR:
                        term -= KR[(t, j)] * alphas[j]
                    if (t, j - 1) in KR:
                        term -= KR[(t, j - 1)] * betas[j]
                    KR[(t, j + 1)][a, b] = sum(term[a, c] * gj1_inv[c, b]
                                              for c in range(r))
                    term_L = mp.zeros(r, r)
                    if t - 1 >= 0 and (t - 1, j) in KL:
                        term_L += KL[(t - 1, j)]
                    if (t, j) in KL:
                        term_L -= alphas[j] * KL[(t, j)]
                    if (t, j - 1) in KL:
                        term_L -= gammas[j] * KL[(t, j - 1)]
                    KL[(t, j + 1)][a, b] = sum(bj1_inv[a, c] * term_L[c, b] for c in range(r))
    return KR, KL
def compute_overlaps_eq56_vs_54(C, m, r, omega, omega_inv, beta_1, gamma_1, KR, KL, N_k):
    """Two-route overlap factors: Eq 54 (cheap) vs Eq 56 (via P_kta)."""
    num_states = m*r
    ZR56 = mp.zeros(num_states, r); ZL56 = mp.zeros(num_states, r)
    ZR54 = mp.zeros(num_states, r); ZL54 = mp.zeros(num_states, r)
    beta_1_inv = mp.inverse(beta_1)
    gamma_1_inv = mp.inverse(gamma_1)
    for k in range(num_states):
        for a in range(r):
            ZR54[k, a] = N_k[k] * sum(beta_1[a, b] * omega[b, k] for b in range(r))
            ZL54[k, a] = (1.0 / N_k[k]) * sum(omega_inv[k, b] * gamma_1[b, a] for b in range(r))
        for a in range(r):
            sum_r = mp.mpf(0); sum_l = mp.mpf(0)
            for t in range(m):
                P_R = [mp.mpf(0)] * r
                P_L = [mp.mpf(0)] * r
                for i in range(t + 1, m + 1):
                    if (t, i) in KR:
                        for b in range(r):
                            for d in range(r):
                                P_R[b] += KR[(t, i)][b, d] * omega[(i - 1) * r + d, k]
                    if (t, i) in KL:
                        for b in range(r):
                            for d in range(r):
                                P_L[b] += omega_inv[k, (i - 1) * r + d] * KL[(t, i)][d, b]
                for b in range(r):
                    P_R[b] *= N_k[k]
                    P_L[b] *= 1.0 / N_k[k]
                for b in range(r):
                    for c in range(r):
                        sum_r += C[t][a, c] * gamma_1_inv[c, b] * P_R[b]
                        sum_l += P_L[b] * beta_1_inv[b, c] * C[t][c, a]
            ZR56[k, a] = sum_r
            ZL56[k, a] = sum_l
    diff_R = max(abs(ZR56[k, a] - ZR54[k, a])
                 for k in range(num_states) for a in range(r))
    diff_L = max(abs(ZL56[k, a] - ZL54[k, a])
                 for k in range(num_states) for a in range(r))
    return ZR56, ZL56, ZR54, ZL54, diff_R, diff_L

def verify_block_lanczos():
    print("VERIFICATION")
    C, E, _ = build_mock(32)
    out = block_lanczos(C, 8)
    r = 2
    print("\n[Check 1] Eq. 30: G_j(1) = gamma_j, B_j(1) = beta_j")
    worst = 0
    for j in range(2, 9):
        eg = max(abs(out["G"][j][1][u, v] - out["gamma"][j][u, v])
                 for u in range(r) for v in range(r))
        eb = max(abs(out["B"][j][1][u, v] - out["beta"][j][u, v])
                 for u in range(r) for v in range(r))
        worst = max(worst, float(eg), float(eb))
    print(f"  worst = {worst:.2e}  "
          f"{'pass' if worst < 1e-20 else 'fail'}")
    print("\n[Check 2] Eq. 37: Ritz values at m=8 vs true e^(-E_k)")
    T8 = assemble_T(out["alpha"], out["beta"], out["gamma"], 8, r)
    vals, _ = eig_values_vectors(T8)
    vals_sorted = sorted(vals, key=lambda v: -mp.re(v))
    max_err = 0
    for k in range(16):
        lam_true = mp.e**(-E[k])
        err = float(abs(vals_sorted[k] - lam_true))
        max_err = max(max_err, err)
    print(f"  max |err| = {max_err:.2e}  "
          f"{'pass' if max_err < 1e-20 else 'fail'}")
    return worst < 1e-20 and max_err < 1e-20
def test_factorization_options():
    """symmetric vs Cholesky vs LU on noiseless mock at m=8"""
    print("Factorization Options (Symmetric, Cholesky, LU)")
    C, _, _ = build_mock(32)
    methods = ["symmetric", "cholesky", "lu"]
    results = {}
    for method in methods:
        out = block_lanczos(C, 8, method=method)
        T8 = assemble_T(out['alpha'], out['beta'], out['gamma'], 8, 2)
        vals, _ = eig_values_vectors(T8)
        vals_sorted = sorted(vals, key=lambda v: -mp.re(v))
        results[method] = vals_sorted
        top3 = [mp.nstr(v, 6) for v in vals_sorted[:3]]
        print(f"  Method '{method:<10}': Top 3 lambda = {top3}")

    print("\n  [Cross-Check] Max |Delta lambda| between methods:")
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            m1, m2 = methods[i], methods[j]
            diffs = [abs(results[m1][k] - results[m2][k]) for k in range(16)]
            max_diff = max(float(d) for d in diffs)
            if max_diff < 1e-20:
                status = "pass"
            elif "lu" in (m1, m2):
                status = "fail (expected for LU breaks beta=gamma^T symmetry)"
            else:
                status = "fail"
            print(f"    {m1:<10} vs {m2:<10}: {max_diff:.2e}  {status}")

def verify_eq56_vs_54():
    print("Eq. 56 (via P_kta) vs Eq. 54 (via omega)")
    C, _, _ = build_mock(32)
    r = 2
    m_max = 3
    out = block_lanczos(C, m_max, method="symmetric")
    beta1, gamma1 = out['beta'][1], out['gamma'][1]
    KR, KL = compute_krylov_K(m_max, r, out['alpha'], out['beta'], out['gamma'])
    all_pass = True
    for m in range(1, m_max + 1):
        T_m = assemble_T(out['alpha'], out['beta'], out['gamma'], m, r)
        vals, V = eig_values_vectors(T_m)
        try:
            Vinv = mp.inverse(V)
        except Exception:
            Vinv = mp.inverse(V + mp.eye(V.rows) * mp.mpf("1e-40"))
        N_k = [mp.mpf(1)] * (m * r)
        ZR56, ZL56, ZR54, ZL54, diff_R, diff_L = compute_overlaps_eq56_vs_54(C, m, r, V, Vinv, beta1, gamma1, KR, KL, N_k)
        ok = (float(diff_R) < 1e-20 and float(diff_L) < 1e-20)
        all_pass = all_pass and ok
        print(f"\n  m={m} ({m*r} states):")
        print(f"    max |Z^R_56 − Z^R_54| = {mp.nstr(diff_R, 3)}")
        print(f"    max |Z^L_56 − Z^L_54| = {mp.nstr(diff_L, 3)}")
        print(f"    {'pass' if ok else 'fail'}")
    print(f"\n  Overall: {'pass' if all_pass else 'fail'}\n")
    return all_pass
if __name__ == "__main__":
    print("krylov_verification.py — library module")
    print("Import from noisy_run_all.py.")

Overwriting krylov_verification.py


In [100]:
%%writefile noise_generator.py
#!/usr/bin/env python3
"""
noise_generator.py mock-noise generation and preprocessing
Functions:
  generate_noisy_data(C_mean_mp, n_cfg=80, n_boot=200, seed=42, rel0=1e-3, E_noise=0.03, rho_t=0.5)
    in  : noiseless C(t), config count, bootstrap count, seed, noise params
    out : samples (n_cfg, Nt, r, r), mean_noisy, boot_means (n_boot, Nt, r, r)
    note: AR(1) time-correlated noise (our choice, paper uses real data)
  to_mpmath_correlator(C_np)
    in  : numpy (Nt, r, r)
    out : list[mp.matrix]
  symmetrize(C_mp)
    in  : list[mp.matrix]
    out : list[mp.matrix]
    eq  : (99) real-symmetric projection
"""
import numpy as np
from mpmath import mp
def generate_noisy_data(C_mean_mp, n_cfg=80, n_boot=200, seed=42, rel0=1e-3, E_noise=0.03, rho_t=0.5):
    rng = np.random.default_rng(seed)
    Nt = len(C_mean_mp)
    r = C_mean_mp[0].rows
    C_mean = np.zeros((Nt, r, r))
    for t in range(Nt):
        for a in range(r):
            for b in range(r):
                C_mean[t, a, b] = float(mp.re(C_mean_mp[t][a, b]))
    g = rng.normal(size=(n_cfg, Nt, r, r))
    g = np.triu(g)
    g = g + np.triu(g, k=1).swapaxes(-1, -2)
    z = np.empty_like(g)
    z[:, 0] = g[:, 0]
    for t in range(1, Nt):
        z[:, t] = rho_t * z[:, t - 1] + np.sqrt(1 - rho_t ** 2) * g[:, t]
    diag0 = np.diag(C_mean[0])
    scale0 = np.sqrt(np.outer(diag0, diag0))
    t_arr = np.arange(Nt)
    sigma = rel0 * np.exp(-E_noise * t_arr)[:, None, None] * scale0[None, :, :]
    samples = C_mean[None] + z * sigma[None]
    mean_noisy = samples.mean(axis=0)
    boot_idx = rng.integers(0, n_cfg, size=(n_boot, n_cfg))
    boot_means = samples[boot_idx].mean(axis=1)
    return samples, mean_noisy, boot_means

def to_mpmath_correlator(C_np):
    Nt, r, _ = C_np.shape
    return [mp.matrix([[mp.mpf(str(C_np[t, a, b])) for b in range(r)]
                       for a in range(r)]) for t in range(Nt)]

def symmetrize(C_mp):
    """Eq 99: C'_ab(t) = (1/2) Re[C_ab(t) + C_ba(t)]"""
    out = []
    for Ct in C_mp:
        r = Ct.rows
        Cs = mp.matrix(r, r)
        for a in range(r):
            for b in range(r):
                Cs[a, b] = (mp.re(Ct[a, b]) + mp.re(Ct[b, a])) / 2
        out.append(Cs)
    return out

if __name__ == "__main__":
    print("noise_generator.py — library module")
    print("Import from noisy_run_all.py.")

Overwriting noise_generator.py


In [101]:
%%writefile hermitian_filter.py
#!/usr/bin/env python3
"""
hermitian_filter.py — Sec V(B) Hermitian-subspace filter (Eqs 107, 108)
analyze_hermitian(omega, omega_inv, lam, beta1, gamma1, r, eps_real=1e-8, eps_imag=1e-8, eps_indep=1e-6)
  in  : eigenvectors (omega, omega_inv), eigenvalues lam,
        initial blocks beta1, gamma1, block size r
  out : list of dicts, one per state k:
          {k, lam, is_real, is_herm, Nk_vals}
        (dZCW_k is NOT computed here — see zcw_filter.py)
  eq  : (107) reality of lam |Im lam / lam| < eps_real
        (108) block norm trick |N_k|^2 = real, positive, a-independent
"""
from mpmath import mp
def analyze_hermitian(omega, omega_inv, lam, beta1, gamma1, r, eps_real=1e-8, eps_imag=1e-8, eps_indep=1e-6):
    n = len(lam)
    results = []
    for k in range(n):
        lam_k = lam[k]
        lam_abs = abs(lam_k)
        is_real = (lam_abs > 0
                   and float(abs(mp.im(lam_k)) / lam_abs) < eps_real)
        if not is_real:
            results.append(dict(k=k, lam=lam_k, is_real=False, is_herm=False, Nk_vals=[]))
            continue
        Nk_vals = []
        for a in range(r):
            num = sum(omega_inv[k, b] * gamma1[b, a] for b in range(r))
            den = sum(beta1[a, c] * omega[c, k] for c in range(r))
            if abs(den) < mp.mpf("1e-40"):
                Nk_vals.append(mp.mpf(0))
                continue
            Nk_vals.append(num / mp.conj(den))
        is_herm = True
        for Nk in Nk_vals:
            if abs(mp.im(Nk)) > eps_imag * max(abs(Nk), mp.mpf("1e-40")):
                is_herm = False
                break
            if mp.re(Nk) <= 0:
                is_herm = False
                break
        if is_herm and len(Nk_vals) > 1:
            ref = mp.re(Nk_vals[0])
            for Nk in Nk_vals[1:]:
                rel = abs(mp.re(Nk) - ref) / max(abs(ref), mp.mpf("1e-40"))
                if float(rel) > eps_indep:
                    is_herm = False
                    break
        results.append(dict(k=k, lam=lam_k, is_real=True, is_herm=is_herm, Nk_vals=Nk_vals))
    return results
if __name__ == "__main__":
    print("hermitian_filter.py — library module")
    print("Import from noisy_run_all.py.")
    print("Eq 107: |Im lam / lam| < eps_float")
    print("Eq 108: |N_k|^2 real, positive, a-independent")

Overwriting hermitian_filter.py


In [102]:
%%writefile zcw_filter.py
#!/usr/bin/env python3
"""
zcw_filter.py — Sec. V.C block ZCW filter (Eqs. 113, 116, 117, 119)
compute_dzcw(omega, omega_inv, r)
  in  : eigenvectors omega, omega_inv, block size r
  out : list of |Delta_k^ZCW| per state k
  eq  : (119)  Delta_k = | sum_a omega_{1ak} (omega^-1)_{k1a} |
find_m_H(per_m_diag, m_max)
  in  : {m -> list of state dicts}, m_max
  out : m_H — last contiguous m where all states physical
  eq  : (116)  m_H = max{m | [k in H & 0 < lambda_k < 1] forall k}
compute_eps_ZCW(per_m_diag, m_H, F_ZCW=10)
  in  : {m -> list of dicts}, m_H, F_ZCW hyperparameter
  out : eps_ZCW cutoff
  eq  : (117)  eps_ZCW = (1/F_ZCW) * min_k Delta_k^ZCW(m_H)
apply_cut(diag_at_m, eps_ZCW)
  in  : list of state dicts at one m, cutoff
  out : sorted list of surviving |lambda_k|
  eq  : (113)  keep only Delta_k^ZCW >= eps_ZCW, real & herm & 0<lam<1
"""
from mpmath import mp
def compute_dzcw(omega, omega_inv, r):
    """Eq 119: block ZCW quantity per state k"""
    n_states = omega.rows
    return [abs(sum(omega[a, k] * omega_inv[k, a] for a in range(r))) for k in range(n_states)]

def find_m_H_strict(per_m_diag, m_max):
    """Eq 116 max over ALL m satisfying the condition"""
    m_H = 0
    for m in range(1, m_max + 1):
        diag = per_m_diag.get(m, [])
        all_phys = bool(diag) and all(
            d['is_real'] and d['is_herm']
            and 0 < float(mp.re(d['lam'])) < 1
            for d in diag)
        if all_phys:
            m_H = m
    return m_H


def find_m_H_contiguous(per_m_diag, m_max):
    """Eq 116 contiguous reading last m in the clean range from m=1"""
    m_H = 0
    found_first_bad = False
    for m in range(1, m_max + 1):
        diag = per_m_diag.get(m, [])
        all_phys = bool(diag) and all(
            d['is_real'] and d['is_herm']
            and 0 < float(mp.re(d['lam'])) < 1
            for d in diag)
        if all_phys and not found_first_bad:
            m_H = m
        elif not all_phys:
            found_first_bad = True
    return m_H

def compute_eps_ZCW(per_m_diag, m_H, F_ZCW=10):
    """Eq 117 with m_H=1 fallback per paper Sec. V(C3)"""
    if m_H <= 1:
        return 1e-2
    dZCW_mH = [float(mp.re(d['dZCW_k'])) for d in per_m_diag[m_H]
               if d['is_real'] and d['is_herm']]
    if not dZCW_mH:
        return 1e-2
    return max(min(dZCW_mH) / F_ZCW, 1e-15)

def apply_cut(diag_at_m, eps_ZCW):
    """Eq 113: keep states with Delta_k^ZCW >= eps_ZCW and physical lam"""
    phys = []
    for d in diag_at_m:
        if not (d['is_real'] and d['is_herm']):
            continue
        if float(mp.re(d['dZCW_k'])) < eps_ZCW:
            continue
        if not (0 < float(mp.re(d['lam'])) < 1):
            continue
        phys.append(float(mp.re(d['lam'])))
    phys.sort(reverse=True)
    return phys

if __name__ == "__main__":
    print("zcw_filter.py — library module")
    print("Import from noisy_run_all.py.")
    print("Eq. 119: Delta_k^ZCW = |sum_a omega_{1ak} (omega^-1)_{k1a}|")
    print("Eq. 116: m_H (contiguous variant)")
    print("Eq. 117: eps_ZCW = (1/F_ZCW) * min Delta_k^ZCW(m_H)")
    print("Eq. 113: cut Delta_k^ZCW < eps_ZCW")

Overwriting zcw_filter.py


In [103]:
%%writefile noisy_run_all.py
#!/usr/bin/env python3
"""
noisy_run_all.py — Section V pipeline orchestrator.

filter_one_bootstrap(C_boot_np, m_max=8, F_ZCW=10)
  in  : bootstrapped C(t) array (Nt, r, r), m_max, F_ZCW
  out : dict{m_H, m_H_strict, m_H_contig, eps_ZCW, survivors, diagnostics, error}
  note: calls analyze_hermitian (Eqs. 107,108) + compute_dzcw (Eq. 119), then find_m_H_* (Eq. 116) + compute_eps_ZCW (Eq. 117) + apply_cut (Eq. 113)
print_m_H_comparison(all_results, m_max)
  in  : bootstrap results, m_max
  out : printed table comparing strict vs contiguous m_H
run_noisy_pipeline(n_cfg=80, n_boot=200, m_max=8, seed=42)
  does: full pipeline + diagnostics
  out : all_results list (or [] on early exit)
main
  does: noiseless check -> factorization test -> eq56 vs 54 -> noisy pipeline
"""
from mpmath import mp
import numpy as np
from lanczos_core import build_mock, block_lanczos, assemble_T, eig_values_vectors
from krylov_verification import verify_block_lanczos, test_factorization_options, verify_eq56_vs_54
from noise_generator import generate_noisy_data, to_mpmath_correlator, symmetrize
from hermitian_filter import analyze_hermitian
from zcw_filter import compute_dzcw, find_m_H_strict, find_m_H_contiguous, compute_eps_ZCW, apply_cut
# Filter one bootstrap
def filter_one_bootstrap(C_boot_np, m_max=8, F_ZCW=10):
    r = 2 # become 2 by 2
    try:
        C_mp = symmetrize(to_mpmath_correlator(C_boot_np))
        out = block_lanczos(C_mp, m_max + 1, method="symmetric")
        beta1, gamma1 = out['beta'][1], out['gamma'][1]
        per_m_diag = {}
        for m in range(1, m_max + 1):
            T_m = assemble_T(out['alpha'], out['beta'], out['gamma'], m, r)
            vals, V = eig_values_vectors(T_m)
            try:
                Vinv = mp.inverse(V)
            except Exception:
                Vinv = mp.inverse(V + mp.eye(V.rows) * mp.mpf("1e-40"))
            diag = analyze_hermitian(V, Vinv, vals, beta1, gamma1, r)
            dzcw = compute_dzcw(V, Vinv, r)
            for k, d in enumerate(diag):
                d['dZCW_k'] = dzcw[k]
            per_m_diag[m] = diag
        m_H_strict = find_m_H_strict(per_m_diag, m_max)
        m_H_contig = find_m_H_contiguous(per_m_diag, m_max)
        m_H = m_H_strict
        eps_ZCW = compute_eps_ZCW(per_m_diag, m_H, F_ZCW)
        survivors = {}
        for m in range(1, m_max + 1):
            survivors[m] = apply_cut(per_m_diag.get(m, []), eps_ZCW)
        return dict(m_H=m_H, m_H_strict=m_H_strict, m_H_contig=m_H_contig, eps_ZCW=eps_ZCW, survivors=survivors, diagnostics=per_m_diag, error=None)
    except Exception as exc:
        return dict(error=f"{exc}", m_H=0, m_H_strict=0, m_H_contig=0, eps_ZCW=1e-2, survivors={}, diagnostics={})
# m_H comparison table
def print_m_H_comparison(all_results, m_max):
    print(f"\n[m_H comparison] strict (literal Eq. 116) vs contiguous")
    strict_vals = [r['m_H_strict'] for r in all_results if not r.get('error')]
    contig_vals = [r['m_H_contig'] for r in all_results if not r.get('error')]
    if not strict_vals:
        print("(no successful bootstraps hence nothing to compare)")
        return
    n_boot = len(strict_vals)
    n_diff = sum(1 for s, c in zip(strict_vals, contig_vals) if s != c)
    print(f"  {'interpretation':<16} | {'median':>6} | {'min':>3} | "
          f"{'max':>3} | {'m=1':>5} | {'m=2':>5} | {'m=3':>5}")
    for name, vals in [("strict (Eq. 116)", strict_vals), ("contiguous", contig_vals)]:
        c1 = sum(1 for v in vals if v == 1)
        c2 = sum(1 for v in vals if v == 2)
        c3 = sum(1 for v in vals if v == 3)
        print(f"  {name:<16} | {int(np.median(vals)):>6} | "
              f"{min(vals):>3} | {max(vals):>3} | "
              f"{c1:>5} | {c2:>5} | {c3:>5}")
    print(f"\n  bootstraps where the two interpretations differ: "
          f"{n_diff} / {n_boot}")
    if n_diff > 0:
        print(f"  (some iteration past the first failure transiently re-cleaned)")
    else:
        print(f"  (interpretations agree on every bootstrap — "
              f"no transient re-cleaning)")
# Full noisy pipeline
def run_noisy_pipeline(n_cfg=80, n_boot=200, m_max=8, seed=42):
    print("HERMITIAN AND BLOCK ZCW FILTERING")
    C_mean_mp, _, _ = build_mock(32)
    _, _, boot_means = generate_noisy_data(C_mean_mp, n_cfg=n_cfg, n_boot=n_boot, seed=seed, rel0=1e-3)
    first = filter_one_bootstrap(boot_means[0], m_max=m_max)
    if first.get('error'):
        print(f"ERROR: {first['error']}")
        return []
    print(f"\n[debug] m_H = {first['m_H']},  ε_ZCW = {first['eps_ZCW']:.4e}")
    print(f"\n  {'m':>3} | {'k':>3} | {'herm':>5} | {'λ':>10} | "
          f"{'Δ^ZCW':>10} | status")
    print(f"  {'-'*3}-+-{'-'*3}-+-{'-'*5}-+-{'-'*10}-+-{'-'*10}-+--------")
    for m in range(1, m_max + 1):
        for d in first['diagnostics'].get(m, []):
            herm = 'Y' if d['is_herm'] else 'n'
            lam_v = mp.nstr(mp.re(d['lam']), 8)
            dzcw = mp.nstr(mp.re(d['dZCW_k']), 5)
            keep = 'keep' if (d['is_real'] and d['is_herm']
                              and 0 < mp.re(d['lam']) < 1
                              and float(mp.re(d['dZCW_k'])) >= first['eps_ZCW']) \
                          else 'cut'
            print(f"  {m:>3} | {d['k']:>3} | {herm:>5} | {lam_v:>10} | "
                  f"{dzcw:>10} | {keep}")
    print(f"\n survivors per m: "
          f"{ {m: len(v) for m, v in first['survivors'].items()} }")
    print(f"\n[run] {n_boot} bootstraps...")
    all_results = []
    for b in range(n_boot):
        all_results.append(filter_one_bootstrap(boot_means[b], m_max=m_max))
        if (b + 1) % 50 == 0:
            print(f"  {b+1}/{n_boot}")
    counts = {m: [] for m in range(1, m_max + 1)}
    for res in all_results:
        for m, phys in res.get('survivors', {}).items():
            counts[m].append(len(phys))
    print(f"\n[results] state counts:")
    print(f"  {'m':>3} | {'mean':>6} | {'med':>4} | {'min':>4} | {'max':>4}")
    for m in range(1, m_max + 1):
        c = np.array(counts[m], dtype=float)
        if c.size == 0:
            print(f"  {m:>3} | {'—':>6} | {'—':>4} | {'—':>4} | {'—':>4}")
        else:
            print(f"  {m:>3} | {c.mean():>6.1f} | {int(np.median(c)):>4} | "
                  f"{int(c.min()):>4} | {int(c.max()):>4}")

    ok = [r for r in all_results if not r.get('error')]
    n_err = len(all_results) - len(ok)
    if ok:
        m_Hs = [r['m_H'] for r in ok]
        eps = [r['eps_ZCW'] for r in ok]
        print(f"\n[diag]  m_H: {min(m_Hs)}..{max(m_Hs)}  "
              f"ε_ZCW: {min(eps):.3e}..{max(eps):.3e}  "
              f"errors: {n_err}/{n_boot}")
    else:
        print(f"\n[diag]  no successful bootstraps; errors: {n_err}/{n_boot}")

    return all_results
# Main
if __name__ == "__main__":
    ok = verify_block_lanczos()
    if not ok:
        print("\n!!!!!!!!!!FIX block_lanczos BEFORE proceeding!!!!!!!!!")
        import sys
        sys.exit(1)
    test_factorization_options()
    verify_eq56_vs_54()
    all_results = run_noisy_pipeline(n_cfg=80, n_boot=200, m_max=8, seed=42)
    print_m_H_comparison(all_results, m_max=8)

Overwriting noisy_run_all.py


In [104]:
!python noisy_run_all.py


VERIFICATION

[Check 1] Eq. 30: G_j(1) = gamma_j, B_j(1) = beta_j
  worst = 4.29e-30  pass

[Check 2] Eq. 37: Ritz values at m=8 vs true e^(-E_k)
  max |err| = 6.18e-28  pass
Factorization Options (Symmetric, Cholesky, LU)
  Method 'symmetric ': Top 3 lambda = ['0.904837', '0.818731', '0.740818']
  Method 'cholesky  ': Top 3 lambda = ['0.904837', '0.818731', '0.740818']
  Method 'lu        ': Top 3 lambda = ['(3.82517 - 5.64884j)', '(3.82517 + 5.64884j)', '(0.904798 - 1.57942e-43j)']

  [Cross-Check] Max |Delta lambda| between methods:
    symmetric  vs cholesky  : 2.93e-28  pass
    symmetric  vs lu        : 6.40e+00  fail (expected for LU breaks beta=gamma^T symmetry)
    cholesky   vs lu        : 6.40e+00  fail (expected for LU breaks beta=gamma^T symmetry)
Eq. 56 (via P_kta) vs Eq. 54 (via omega)

  m=1 (2 states):
    max |Z^R_56 − Z^R_54| = 4.59e-41
    max |Z^L_56 − Z^L_54| = 4.59e-41
    pass

  m=2 (4 states):
    max |Z^R_56 − Z^R_54| = 3.44e-40
    max |Z^L_56 − Z^L_54| = 3.